In [ ]:
import duckdb
import osmnx as ox
import geopandas as gpd

# 1. Pull Roads using OSMnx (Using raw coordinates instead of a text name)
# Center of Kelapa Gading: Lat -6.155, Lon 106.900
center_point = (-6.155, 106.900)

# Pull all roads within a 2000 meter (2km) radius of that exact point
graph = ox.graph_from_point(center_point, dist=2000, network_type='drive')
nodes, edges = ox.graph_to_gdfs(graph)

# 2. Connect to DuckDB and load spatial extensions
conn = duckdb.connect()
conn.execute("INSTALL spatial; LOAD spatial; INSTALL httpfs; LOAD httpfs;")

# 3. Pull Overture POIs for that exact same geographic window
query = """
SELECT
    names.primary as name,
    categories.primary as category,
    ST_AsText(geometry) as geometry_wkt
FROM read_parquet('s3://overturemaps-us-west-2/release/2026-06-17.0/theme=places/type=place/*')
WHERE bbox.xmin > 106.88 AND bbox.xmax < 106.92
  AND bbox.ymin > -6.17 AND bbox.ymax < -6.14
"""
pois_df = conn.execute(query).df()

In [ ]:
from shapely import wkt

# Convert the clean text strings into actual shapely geometries
pois_df['geometry'] = pois_df['geometry_wkt'].apply(wkt.loads)
pois_gdf = gpd.GeoDataFrame(pois_df, geometry='geometry', crs="EPSG:4326")

# Reproject both datasets to a metric CRS for accurate distance calculations
pois_gdf = pois_gdf.to_crs(epsg=32748)
edges = edges.to_crs(epsg=32748)

# Find the distance to the nearest road edge
nearest_roads = gpd.sjoin_nearest(pois_gdf, edges, how="left", distance_col="distance_to_road")

In [ ]:
# Filter for POIs further than 50 meters from a road
anomalies = nearest_roads[nearest_roads['distance_to_road'] > 100]

# Display how many anomalies you found
print(f"Total POIs analyzed: {len(nearest_roads)}")
print(f"Number of unmapped/anomalous POIs found: {len(anomalies)}")


In [ ]:
import matplotlib.pyplot as plt

# Set up a plot area
fig, ax = plt.subplots(figsize=(12, 12))

# 1. Plot all the OSM roads as faint gray lines
edges.plot(ax=ax, color='lightgray', linewidth=1, label='OSM Roads')

# 2. Plot normal POIs (close to roads) as small blue dots
normal_pois = nearest_roads[nearest_roads['distance_to_road'] <= 100]
normal_pois.plot(ax=ax, color='blue', markersize=5, alpha=0.3, label='Normal POIs')

# 3. Plot the anomalies as large, bright red dots so they stand out!
if len(anomalies) > 0:
    anomalies.plot(ax=ax, color='red', markersize=20, alpha=0.7, label='Missing Road Candidates')

# Customizing the map appearance
plt.title("Missing Road Geometry Detection - Kelapa Gading", fontsize=14)
plt.legend()
plt.axis('off')
plt.show()

In [ ]:
from sklearn.cluster import DBSCAN
import numpy as np

# 1. Extract the raw X and Y coordinates (in meters) from your anomalies
coords = np.array(list(zip(anomalies.geometry.x, anomalies.geometry.y)))

# 2. Run DBSCAN: eps=100 meters, min_samples=5 points to make a cluster
db = DBSCAN(eps=100, min_samples=5).fit(coords)

# 3. Add the cluster labels back to your GeoDataFrame
anomalies['cluster_id'] = db.labels_

# 4. Filter out the noise (-1 indicates noise in DBSCAN)
valid_clusters = anomalies[anomalies['cluster_id'] != -1]

print(f"Filtered out noise points. Found {valid_clusters['cluster_id'].nunique()} distinct unmapped zones!")

In [ ]:
# 1. Group the points by their cluster_id and merge their geometries together
cluster_multipoints = valid_clusters.groupby('cluster_id')['geometry'].apply(lambda points: points.unary_union)

# 2. Draw a "Convex Hull" (a shrink-wrapped polygon boundary) around each cluster
unmapped_zones_polygons = gpd.GeoDataFrame(geometry=cluster_multipoints.convex_hull, crs=valid_clusters.crs)

# 3. Export these beautiful polygons so you can drop them into QGIS!
unmapped_zones_polygons.to_crs(epsg=4326).to_file("39_unmapped_zones.geojson", driver="GeoJSON")

print("Successfully generated and exported 39 Unmapped Zone Polygons!")

In [ ]:
import numpy as np
from google.colab import data_table

# ... (Previous code: crs conversion, lat/lon extraction, renaming columns) ...

# 4.5 CREATE THE DERIVED PRIORITY METRIC
# Define the conditions based on the severity of the distance
conditions = [
    (dashboard_df['Distance_to_Road_m'] >= 200),
    (dashboard_df['Distance_to_Road_m'] >= 150) & (dashboard_df['Distance_to_Road_m'] < 200),
    (dashboard_df['Distance_to_Road_m'] < 150)
]

# Define the text labels that match those conditions
choices = ['High', 'Medium', 'Low']

# Apply the logic to create the brand new 'Priority_Level' column
dashboard_df['Priority_Level'] = np.select(conditions, choices, default='Low')

# ... (Continue to export step) ...

dashboard_df